# 📐 MACHOTE — Regresión Polinomial

## ¿Qué es y en qué se diferencia de la lineal?

| | Regresión Lineal | Regresión Polinomial |
|---|---|---|
| Ecuación | `Y = wX + b` | `Y = w1X + w2X² + w3X³ + b` |
| Forma | Línea recta | Curva |
| Cuándo usarla | Datos con relación recta | Datos con forma de curva |
| Librería nueva | — | `PolynomialFeatures` |

**La clave:** `PolynomialFeatures` transforma tus variables X originales
agregando columnas con potencias (X², X³, etc.), y luego `LinearRegression`
trabaja igual que siempre sobre esas columnas nuevas.

```
X original: [edad]  →  PolynomialFeatures(degree=2)  →  [1, edad, edad²]
                                                           ↑ término de sesgo
```

## Parámetro más importante: `degree` (grado)
```
degree=1  →  línea recta (= regresión lineal normal)
degree=2  →  parábola  (el más común para empezar)
degree=3  →  curva cúbica
degree=N  →  OJO: a mayor grado, más riesgo de overfitting
```

## Overfitting vs Underfitting
```
Underfitting  →  degree muy bajo, el modelo es demasiado simple, no aprende
Buen ajuste   →  degree adecuado, generaliza bien con datos nuevos
Overfitting   →  degree muy alto, memoriza los datos de entrenamiento
                  pero falla con datos nuevos (R² train alto, R² test bajo)
```

In [ ]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================
# Todo lo de regresión lineal +
# PolynomialFeatures → transforma X a [X, X², X³...]
# Pipeline          → encadena transformación + modelo en un solo paso

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures   # ← NUEVO
from sklearn.pipeline import Pipeline                  # ← NUEVO (opcional pero recomendado)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ============================================================
# 2. CARGAR Y PREPARAR DATOS
# ============================================================
# Igual que siempre: cargar CSV, seleccionar columnas, encodear.
# Aquí usamos insurance.csv de ejemplo.
# ↓↓ CAMBIAR según el dataset del ejercicio ↓↓

df = pd.read_csv("insurance.csv")

# Codificar categóricas
df['sex']    = df['sex'].map({'male': 1, 'female': 0})
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})
df = pd.get_dummies(df, columns=['region'], dtype=int)

print(df.head())
print("Columnas:", df.columns.tolist())

In [ ]:
# ============================================================
# 3. SELECCIONAR X e y
# ============================================================
# Para regresión polinomial puedes usar 1 o varias variables.
# Con 1 variable es más fácil visualizar la curva resultante.
# ↓↓ CAMBIAR según el ejercicio ↓↓

# Opción A: 1 sola variable (fácil de graficar)
X = df[['age']].values          # shape: (n, 1)
y = df['charges'].values        # shape: (n,)

# Opción B: varias variables (descomentar si el ejercicio lo pide)
# X = df[['age', 'bmi', 'smoker']].values
# y = df['charges'].values

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

In [ ]:
# ============================================================
# 4. NORMALIZACIÓN MIN-MAX
# ============================================================
# Igual que en regresión lineal: escalar todo a [0,1].
# Guardar min/max ANTES de normalizar para desnormalizar después.

X_min, X_max = X.min(axis=0), X.max(axis=0)
y_min, y_max = y.min(), y.max()

X_norm = (X - X_min) / (X_max - X_min)
y_norm = (y - y_min) / (y_max - y_min)

print("X normalizada — min:", X_norm.min(), "max:", X_norm.max())
print("y normalizada — min:", round(y_norm.min(), 2), "max:", round(y_norm.max(), 2))

In [ ]:
# ============================================================
# 5. DIVIDIR EN ENTRENAMIENTO Y PRUEBA
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm,
    test_size=0.2,
    random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}")

In [ ]:
# ============================================================
# 6. CREAR MODELO POLINOMIAL
# ============================================================
# HAY DOS FORMAS de hacer esto — ambas válidas:

# ── FORMA A: Manual (paso a paso) ──────────────────────────
# Primero transformas X, luego entrenas el modelo por separado.
# Más didáctica para entender qué hace PolynomialFeatures.

GRADO = 2    # ← CAMBIAR: probar con 2, 3, 4 y comparar R²

poly        = PolynomialFeatures(degree=GRADO, include_bias=False)
X_train_poly = poly.fit_transform(X_train)  # transforma train
X_test_poly  = poly.transform(X_test)       # transforma test con el mismo poly

print(f"X_train original shape: {X_train.shape}")
print(f"X_train polinomial shape: {X_train_poly.shape}")
print("→ Agregó columnas: X² (y X³ si degree=3, etc.)")

modelo_A = LinearRegression()
modelo_A.fit(X_train_poly, y_train)


# ── FORMA B: Pipeline (todo en uno) ────────────────────────
# Encadena PolynomialFeatures + LinearRegression en un solo objeto.
# Más limpio, menos código, mismo resultado.

modelo_B = Pipeline([
    ('poly',      PolynomialFeatures(degree=GRADO, include_bias=False)),
    ('regresion', LinearRegression())
])
modelo_B.fit(X_train, y_train)

print("\nAmbos modelos entrenados.")

In [ ]:
# ============================================================
# 7. EVALUAR MÉTRICAS
# ============================================================
# Las métricas son exactamente iguales que en regresión lineal.
# La diferencia: ahora predices con X_test_poly (Forma A)
#                o directamente con X_test (Forma B / Pipeline).

# --- Forma A ---
pred_A = modelo_A.predict(X_test_poly)

# --- Forma B (Pipeline) ---
pred_B = modelo_B.predict(X_test)

def mostrar_metricas(nombre, y_test, predicciones):
    mae = mean_absolute_error(y_test, predicciones)
    mse = mean_squared_error(y_test, predicciones)
    r2  = r2_score(y_test, predicciones)
    print(f"\n── {nombre} ──")
    print(f"  MAE: {round(mae, 4)}")
    print(f"  MSE: {round(mse, 4)}")
    print(f"  R²:  {round(r2,  4)}  → explica el {round(r2*100, 1)}% de la variación")

print(f"MÉTRICAS (degree={GRADO})")
mostrar_metricas("Forma A (manual)",   y_test, pred_A)
mostrar_metricas("Forma B (Pipeline)", y_test, pred_B)

In [ ]:
# ============================================================
# 8. COMPARAR GRADOS — encontrar el mejor
# ============================================================
# Probamos varios grados y comparamos su R² en test.
# El mejor grado es el que da R² más alto en TEST (no en train).
# Si R² train sube mucho pero R² test baja → overfitting.

print(f"{'Grado':<8} {'R² Train':>10} {'R² Test':>10}")
print("-" * 32)

mejor_r2    = -999
mejor_grado = 1

for grado in range(1, 7):
    modelo_cmp = Pipeline([
        ('poly', PolynomialFeatures(degree=grado, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    modelo_cmp.fit(X_train, y_train)

    r2_train = r2_score(y_train, modelo_cmp.predict(X_train))
    r2_test  = r2_score(y_test,  modelo_cmp.predict(X_test))

    alerta = " ← OVERFITTING" if r2_train - r2_test > 0.1 else ""
    print(f"{grado:<8} {r2_train:>10.4f} {r2_test:>10.4f}{alerta}")

    if r2_test > mejor_r2:
        mejor_r2    = r2_test
        mejor_grado = grado

print(f"\n→ Mejor grado: {mejor_grado} (R² test = {round(mejor_r2, 4)})")

In [ ]:
# ============================================================
# 9. GRÁFICA — curva polinomial vs datos reales
# ============================================================
# Solo funciona bien visualmente con 1 variable de entrada.
# Si el ejercicio usa varias variables, omite esta celda.

# Modelo con el mejor grado encontrado
modelo_final = Pipeline([
    ('poly', PolynomialFeatures(degree=mejor_grado, include_bias=False)),
    ('reg',  LinearRegression())
])
modelo_final.fit(X_norm, y_norm)    # entrenar con TODOS los datos para graficar

# Curva suave para la línea del modelo
X_linea      = np.linspace(0, 1, 300).reshape(-1, 1)
y_linea_norm = modelo_final.predict(X_linea)

# Desnormalizar para mostrar valores reales
X_linea_real = X_linea * (X_max - X_min) + X_min
y_linea_real = y_linea_norm * (y_max - y_min) + y_min
X_real       = X_norm * (X_max - X_min) + X_min
y_real       = y_norm * (y_max - y_min) + y_min

plt.figure(figsize=(10, 5))
plt.scatter(X_real, y_real, alpha=0.3, s=15, color='steelblue', label='Datos reales')
plt.plot(X_linea_real, y_linea_real, color='orangered', linewidth=2,
         label=f'Modelo polinomial (degree={mejor_grado})')
plt.title(f'Regresión Polinomial — degree={mejor_grado}', fontsize=14)
plt.xlabel('Edad', fontsize=12)
plt.ylabel('Cargos (USD)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10. PREDECIR NUEVO VALOR
# ============================================================
# Normalizar el dato nuevo con los mismos min/max del dataset.
# El Pipeline aplica PolynomialFeatures automáticamente.

# ↓↓ CAMBIAR el valor según el ejercicio ↓↓
edad_nueva = 35

# Normalizar
edad_norm_nueva = (edad_nueva - X_min) / (X_max - X_min)

# Predecir (normalizado)
pred_norm = modelo_final.predict([[edad_norm_nueva]])

# Desnormalizar
pred_real = (pred_norm[0] * (y_max - y_min)) + y_min

print(f"Edad ingresada:             {edad_nueva} años")
print(f"Costo estimado del seguro:  ${round(pred_real, 2)} USD")

---
## 📌 Resumen rápido — qué cambiar según el ejercicio

| Qué | Dónde | Qué poner |
|---|---|---|
| Dataset | Celda 2 | `pd.read_csv('tu_archivo.csv')` |
| Variable(s) X | Celda 3 | columnas que uses como entrada |
| Variable Y | Celda 3 | columna que quieres predecir |
| Grado inicial | Celda 6 | `GRADO = 2` (cambiar y experimentar) |
| Valor a predecir | Celda 10 | el dato del cliente/ejemplo |

## 📌 Errores comunes

- **Usar `poly.fit_transform` en test** → siempre `transform` en test, `fit_transform` solo en train
- **Olvidar desnormalizar** → la predicción sale en [0,1] en lugar de dólares/kg/lo que sea
- **Grado muy alto** → overfitting: R² train muy alto pero R² test bajo
- **No guardar min/max antes de normalizar** → no puedes desnormalizar correctamente

## 📌 Diferencia fit_transform vs transform
```python
poly.fit_transform(X_train)  # aprende los parámetros Y transforma → solo en TRAIN
poly.transform(X_test)       # solo transforma usando lo que ya aprendió → en TEST
```